In [23]:
import pandas as pd

# Load datasets
weather_data = pd.read_csv('Weather report (2019 -2024).csv', skiprows=1)
forecast_data = pd.read_csv('Forcast Weather.csv')

# Standardize column names
weather_data.columns = [
    "Date", "Max Temp (°F)", "Avg Temp (°F)", "Min Temp (°F)",
    "Max Dew Point (°F)", "Avg Dew Point (°F)", "Min Dew Point (°F)",
    "Max Humidity (%)", "Avg Humidity (%)", "Min Humidity (%)",
    "Max Wind Speed (mph)", "Avg Wind Speed (mph)", "Min Wind Speed (mph)",
    "Max Pressure (in)", "Avg Pressure (in)", "Min Pressure (in)",
    "Precipitation (in)"
]

# Fill missing values and ensure 'Date' is in datetime format
weather_data.fillna(0, inplace=True)
weather_data['Date'] = pd.to_datetime(weather_data['Date'], format="%m/%d/%y", errors='coerce')
weather_data = weather_data[weather_data['Date'].notna()]

# Convert numeric columns to proper types
numeric_columns = [
    "Max Temp (°F)", "Avg Temp (°F)", "Min Temp (°F)",
    "Max Dew Point (°F)", "Avg Dew Point (°F)", "Min Dew Point (°F)",
    "Max Humidity (%)", "Avg Humidity (%)", "Min Humidity (%)",
    "Max Wind Speed (mph)", "Avg Wind Speed (mph)", "Min Wind Speed (mph)",
    "Max Pressure (in)", "Avg Pressure (in)", "Min Pressure (in)",
    "Precipitation (in)"
]
weather_data[numeric_columns] = weather_data[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Add Year, Month, and Day columns
weather_data['Year'] = weather_data['Date'].dt.year
weather_data['Month'] = weather_data['Date'].dt.month_name()
weather_data['Day'] = weather_data['Date'].dt.day

forecast_data.columns = [
    "Date", "Max Temp (°F)", "Avg Temp (°F)", "Min Temp (°F)",
    "Max Dew Point (°F)", "Avg Dew Point (°F)", "Min Dew Point (°F)",
    "Max Humidity (%)", "Avg Humidity (%)", "Min Humidity (%)",
    "Max Wind Speed (mph)", "Avg Wind Speed (mph)", "Min Wind Speed (mph)",
    "Max Pressure (in)", "Avg Pressure (in)", "Min Pressure (in)",
    "Precipitation (in)"
]
forecast_data.fillna(0, inplace=True)
forecast_data['Date'] = pd.to_datetime(forecast_data['Date'], errors='coerce')
forecast_data = forecast_data[forecast_data['Date'].notna()]
forecast_data = forecast_data[forecast_data['Date'] > pd.Timestamp.now().normalize()]

# Function to get valid menu input from the user
def get_valid_menu_input(prompt, min_val, max_val):
    while True:
        try:
            value = int(input(prompt))
            if min_val <= value <= max_val:
                return value
            else:
                print(f"Invalid Entry. Please enter a number between {min_val} and {max_val}.")
        except ValueError:
            print("Invalid Entry. Please enter a valid number.")

# Menu options
def display_current_weather():
    current_date = pd.Timestamp.now().normalize()
    current_data = weather_data.loc[weather_data['Date'] == current_date]
    if current_data.empty:
        print("No current weather data available for today. Showing the latest available data:")
        latest_data = weather_data.iloc[-1]
        print(f"Date: {latest_data['Date'].strftime('%A, %Y-%m-%d')}")
        print(f"Temperature: {latest_data['Avg Temp (°F)']}°F")
        print(f"Humidity: {latest_data['Avg Humidity (%)']}%")
        print(f"Wind Speed: {latest_data['Avg Wind Speed (mph)']} mph")
    else:
        current = current_data.iloc[0]
        print(f"Date: {current['Date'].strftime('%A, %Y-%m-%d')}")
        print(f"Temperature: {current['Avg Temp (°F)']}°F")
        print(f"Humidity: {current['Avg Humidity (%)']}%")
        print(f"Wind Speed: {current['Avg Wind Speed (mph)']} mph")

def display_monthly_weather():
    last_month = pd.Timestamp.now() - pd.DateOffset(months=1)
    monthly_data = weather_data[
        (weather_data['Year'] == last_month.year) &
        (weather_data['Month'] == last_month.strftime('%B'))
    ]
    if monthly_data.empty:
        print("No data for the last month.")
    else:
        avg_temp = pd.to_numeric(monthly_data['Avg Temp (°F)'], errors='coerce').mean()
        avg_humidity = pd.to_numeric(monthly_data['Avg Humidity (%)'], errors='coerce').mean()
        avg_wind_speed = pd.to_numeric(monthly_data['Avg Wind Speed (mph)'], errors='coerce').mean()
        print(f"\nAverage Weather Data for {last_month.strftime('%B').upper()}/{last_month.year}")
        print(f"Average Temperature: {avg_temp:.2f}°F")
        print(f"Average Humidity: {avg_humidity:.2f}%")
        print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

def display_historical_weather():
    try:
        start_date = pd.to_datetime(input("Enter the start date (YYYY-MM-DD): "))
        end_date = pd.to_datetime(input("Enter the end date (YYYY-MM-DD): "))
    except ValueError:
        print("Invalid date format. Please enter dates in the format YYYY-MM-DD.")
        return

    if (end_date - start_date).days > 7:
        print("Error: Maximum date range is 7 days.")
        return

    historical_data = weather_data[
        (weather_data['Date'] >= start_date) & (weather_data['Date'] <= end_date)
    ]
    if historical_data.empty:
        print("No historical data available for the given range.")
    else:
        avg_temp = pd.to_numeric(historical_data['Avg Temp (°F)'], errors='coerce').mean()
        avg_humidity = pd.to_numeric(historical_data['Avg Humidity (%)'], errors='coerce').mean()
        avg_wind_speed = pd.to_numeric(historical_data['Avg Wind Speed (mph)'], errors='coerce').mean()
        print(f"\nHistorical Weather Data from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
        print(f"Average Temperature: {avg_temp:.2f}°F")
        print(f"Average Humidity: {avg_humidity:.2f}%")
        print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

def display_forecast_weather():
    try:
        forecast_date = pd.to_datetime(input("Enter the forecast date (YYYY-MM-DD): "))
    except ValueError:
        print("Invalid date format. Please enter dates in the format YYYY-MM-DD.")
        return

    if forecast_date <= pd.Timestamp.now().normalize():
        print("Error: Please enter a date after today's date for the forecast.")
        return

    forecast_data_for_date = forecast_data[forecast_data['Date'] == forecast_date]
    if forecast_data_for_date.empty:
        print("No forecast data available for the given date.")
    else:
        avg_temp = pd.to_numeric(forecast_data_for_date['Avg Temp (°F)'], errors='coerce').mean()
        avg_humidity = pd.to_numeric(forecast_data_for_date['Avg Humidity (%)'], errors='coerce').mean()
        avg_wind_speed = pd.to_numeric(forecast_data_for_date['Avg Wind Speed (mph)'], errors='coerce').mean()
        print(f"\nForecast Weather Data for {forecast_date.strftime('%A, %Y-%m-%d')}")
        print(f"Average Temperature: {avg_temp:.2f}°F")
        print(f"Average Humidity: {avg_humidity:.2f}%")
        print(f"Average Wind Speed: {avg_wind_speed:.2f} mph")

# Main menu function to display options and link to weather dashboard
def main_menu():
    while True:
        print("\nWeather Dashboard Main Menu")
        print("1. Current Weather")
        print("2. Monthly Weather")
        print("3. Historical Weather")
        print("4. Forecast Weather")
        print("5. Exit")

        menu_input = get_valid_menu_input("Please enter menu selection (1 - 5): ", 1, 5)

        if menu_input == 1:
            display_current_weather()
        elif menu_input == 2:
            display_monthly_weather()
        elif menu_input == 3:
            display_historical_weather()
        elif menu_input == 4:
            display_forecast_weather()
        elif menu_input == 5:
            print("Exiting the Weather Dashboard.")
            break

# Run the main menu
if __name__ == "__main__":
    main_menu()



Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  1


Date: Tuesday, 2024-11-26
Temperature: 48.8°F
Humidity: 33.4%
Wind Speed: 7.9 mph

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  2



Average Weather Data for OCTOBER/2024
Average Temperature: 76.53°F
Average Humidity: 20.65%
Average Wind Speed: 7.16 mph

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


...

Invalid Entry. Please enter a valid number.


Please enter menu selection (1 - 5):  4
Enter the forecast date (YYYY-MM-DD):  2024-12-02



Forecast Weather Data for Monday, 2024-12-02
Average Temperature: 56.80°F
Average Humidity: 44.40%
Average Wind Speed: 8.80 mph

Weather Dashboard Main Menu
1. Current Weather
2. Monthly Weather
3. Historical Weather
4. Forecast Weather
5. Exit


Please enter menu selection (1 - 5):  5


Exiting the Weather Dashboard.


In [17]:
import pandas as pd

def load_current_weather():
    """Load and display current weather data from a CSV file."""
    try:
        current_weather = pd.read_csv('CURRENT WEATHER REPORT.csv')
        if all(col in current_weather.columns for col in ['Date', 'Temperature (°F)', 'Dew Point (°F)', 'Humidity (%)', 'Wind Speed (mph)', 'Pressure (in)', 'Precipitation (in)']):
            print("Current Weather Data:")
            for index, row in current_weather.iterrows():
                if pd.notna(row['Date']):
                    print(f"Date: {row['Date']}")
                    print(f"Max Temperature: {row['Temperature (°F)'].split()[0]}°F")
                    print(f"Avg Temperature: {row['Temperature (°F)'].split()[1]}°F")
                    print(f"Min Temperature: {row['Temperature (°F)'].split()[2]}°F")
                    print(f"Dew Point: {row['Dew Point (°F)']}°F")
                    print(f"Humidity: {row['Humidity (%)']}%")
                    print(f"Wind Speed: {row['Wind Speed (mph)']} mph")
                    print(f"Pressure: {row['Pressure (in)']} in")
                    print(f"Precipitation: {row['Precipitation (in)']} in\n")
                else:
                    print("Warning: Missing date in current weather entry.")
        else:
            print("Error: Required columns are missing in the current weather data.")
    except Exception as e:
        print(f"Error loading current weather data: {e}")

def load_monthly_weather():
    """Load and display monthly weather data from a CSV file."""
    try:
        monthly_weather = pd.read_csv('Monthly weather report (2019 - 2024).csv')
        if 'Month' in monthly_weather.columns:
            print("Monthly Weather Data:")
            for index, row in monthly_weather.iterrows():
                if pd.notna(row['Month']):
                    print(f"Month: {row['Month']}")
                    print(f"Average Temperature: {row['Avg Temperature']}°F")
                    print(f"Average Humidity: {row['Avg Humidity']}%")
                    print(f"Total Rainfall: {row['Total Rainfall']} inches\n")
                else:
                    print("Warning: Missing data in monthly weather entry.")
        else:
            print("Error: Required columns are missing in the monthly weather data.")
    except Exception as e:
        print(f"Error loading monthly weather data: {e}")

def load_historical_weather():
    """Load and display historical weather data from a CSV file."""
    try:
        historical_weather = pd.read_csv('Historical weather report (2019 - 2024).csv')
        if all(col in historical_weather.columns for col in ['Date', 'Temperature (°F)', 'Dew Point (°F)', 'Humidity (%)', 'Wind Speed (mph)', 'Pressure (in)', 'Precipitation (in)']):
            print("Historical Weather Data:")
            for index, row in historical_weather.iterrows():
                if pd.notna(row['Date']):
                    print(f"Date: {row['Date']}")
                    print(f"Max Temperature: {row['Temperature (°F)'].split()[0]}°F")
                    print(f"Avg Temperature: {row['Temperature (°F)'].split()[1]}°F")
                    print(f"Min Temperature: {row['Temperature (°F)'].split()[2]}°F")
                    print(f"Dew Point: {row['Dew Point (°F)']}°F")
                    print(f"Humidity: {row['Humidity (%)']}%")
                    print(f"Wind Speed: {row['Wind Speed (mph)']} mph")
                    print(f"Pressure: {row['Pressure (in)']} in")
                    print(f"Precipitation: {row['Precipitation (in)']} in\n")
                else:
                    print("Warning: Missing date in historical weather entry.")
        else:
            print("Error: Required columns are missing in the historical weather data.")
    except Exception as e:
        print(f"Error loading historical weather data: {e}")

#def load_forecast_weather():
#    """Load and display forecast weather data from a CSV file."""
#    try:
 #       forecast_weather = pd.read_csv('forecast_weather.csv')
  #      if all(col in forecast_weather.columns for col in ['Date', 'Predicted Temperature (°F)', 'Predicted Dew Point (°F)', 'Predicted Humidity (%)', 'Predicted Wind Speed (mph)', 'Predicted Pressure (in)', 'Predicted Precipitation (in)']):
   #         print("Forecast Weather Data:")
    #        for index, row in forecast_weather.iterrows():
     #           if pd.notna(row['Date']):
      #              print(f"Date: {row['Date']}")
       #             print(f"Predicted Max Temperature: {row['Predicted Temperature (°F)'].split()[0]}°F")
        #            print(f"Predicted Avg Temperature: {row['Predicted Temperature (°F)'].split()[1]}°F")
         #           print(f"Predicted Min Temperature: {row['Predicted Temperature (°F)'].split()[2]}°F")
          #          print(f"Predicted Dew Point: {row['Predicted Dew Point (°F)']}°F")
           #         print(f"Predicted Humidity: {row['Predicted Humidity (%)']}%")
            #        print(f"Predicted Wind Speed: {row['Predicted Wind Speed (mph)']} mph")
             #       print(f"Predicted Pressure: {row['Predicted Pressure (in)']} in")
              #      print(f"Predicted Precipitation: {row['Predicted Precipitation (in)']} in\n")
               # else:
                #    print("Warning: Missing date in forecast weather entry.")
        #else:
           # print("Error: Required columns are missing in the forecast weather data.")
    #except Exception as e:
     #   print(f"Error loading forecast weather data: {e}")

def main_menu():
    """Display the main menu and handle user input."""
    while True:
        print("\nWeather Data Management System")
        print("1. Load Current Weather")
        print("2. Load Monthly Weather")
        print("3. Load Historical Weather")
        print("4. Load Forecast Weather")
        print("5. Exit")
        
        choice = input("Please select an option (1-5): ")
        
        if choice == '1':
            load_current_weather()
        elif choice == '2':
            load_monthly_weather()
        elif choice == '3':
            load_historical_weather()
        elif choice == '4':
            load_forecast_weather()
        elif choice == '5':
            print("Exiting the program. Goodbye!")
            break
        else:
            print("Invalid choice. Please select a valid option.")

if __name__ == "__main__":
    main_menu()


Weather Data Management System
1. Load Current Weather
2. Load Monthly Weather
3. Load Historical Weather
4. Load Forecast Weather
5. Exit


Please select an option (1-5):  1


Current Weather Data:
Date: 2024-11-01
Max Temperature: 71°F
Error loading current weather data: list index out of range

Weather Data Management System
1. Load Current Weather
2. Load Monthly Weather
3. Load Historical Weather
4. Load Forecast Weather
5. Exit


Please select an option (1-5):  2


Error: Required columns are missing in the monthly weather data.

Weather Data Management System
1. Load Current Weather
2. Load Monthly Weather
3. Load Historical Weather
4. Load Forecast Weather
5. Exit


Please select an option (1-5):  5


Exiting the program. Goodbye!


In [21]:
import pandas as pd

def load_current_weather():
    """Load and display current weather data from a CSV file."""
    try:
        current_weather = pd.read_csv('CURRENT WEATHER REPORT.csv')
        # Check if required columns are present
        if all(col in current_weather.columns for col in ['Date', 'Temperature (°F)', 'Dew Point (°F)', 'Humidity (%)', 'Wind Speed (mph)', 'Pressure (in)', 'Precipitation (in)']):
            print("Current Weather Data:")
            for index, row in current_weather.iterrows():
                if pd.notna(row['Date']):
                    print(f"Date: {row['Date']}")
                    
                    # Ensure the temperature data is in the expected format
                    temp_values = row['Temperature (°F)'].split()
                    if len(temp_values) >= 3:  # Check if we have at least 3 values
                        print(f"Max Temperature: {temp_values[0]}°F")
                        print(f"Avg Temperature: {temp_values[1]}°F")
                        print(f"Min Temperature: {temp_values[2]}°F")
                    else:
                        print("Warning: Temperature data is not in the expected format.")
                    
                    print(f"Dew Point: {row['Dew Point (°F)']}°F")
                    print(f"Humidity: {row['Humidity (%)']}%")
                    print(f"Wind Speed: {row['Wind Speed (mph)']} mph")
                    print(f"Pressure: {row['Pressure (in)']} in")
                    print(f"Precipitation: {row['Precipitation (in)']} in\n")
                else:
                    print("Warning: Missing date in current weather entry.")
        else:
            print("Error: Required columns are missing in the current weather data.")
    except Exception as e:
        print(f"Error loading current weather data: {e}")

def load_monthly_weather():
    """Load and display monthly weather data from a CSV file."""
    try:
        monthly_weather = pd.read_csv('Monthly weather report (2019 - 2024).csv')
        if 'Month' in monthly_weather.columns:
            print("Monthly Weather Data:")
            for index, row in monthly_weather.iterrows():
                if pd.notna(row['Month']):
                    print(f"Month: {row['Month']}")
                    print(f"Average Temperature: {row['Avg Temperature']}°F")
                    print(f"Average Humidity: {row['Avg Humidity']}%")
                    print(f"Total Rainfall: {row['Total Rainfall']} inches\n")
                else:
                    print("Warning: Missing data in monthly weather entry.")
        else:
            print("Error: Required columns are missing in the monthly weather data.")
    except Exception as e:
        print(f"Error loading monthly weather data: {e}")

def load_historical_weather():
    """Load and display historical weather data from a CSV file."""
    try:
        historical_weather = pd.read_csv('Historical weather report (2019 - 2024).csv')
        if all(col in historical_weather.columns for col in ['Date', 'Temperature (°F)', 'Dew Point (°F)', 'Humidity (%)', 'Wind Speed (mph)', 'Pressure (in)', 'Precipitation (in)']):
            print("Historical Weather Data:")
            for index, row in historical_weather.iterrows():
                if pd.notna(row['Date']):
                    print(f"Date: {row['Date']}")
                    
                    # Ensure the temperature data is in the expected format
                    temp_values = row['Temperature (°F)'].split()
                    if len(temp_values) >= 3:  # Check if we have at least 3 values
                        print(f"Max Temperature: {temp_values[0]}°F")
                        print(f"Avg Temperature: {temp_values[1]}°F")
                        print(f"Min Temperature: {temp_values[2]}°F")
                    else:
                        print("Warning: Temperature data is not in the expected format.")
                    
                    print(f"Dew Point: {row['Dew Point (°F)']}°F")
                    print(f"Humidity: {row['Humidity (%)']}%")
                    print(f"Wind Speed: {row['Wind Speed (mph)']} mph")
                    print(f"Pressure: {row['Pressure (in)']} in")
                    print(f"Precipitation: {row['Precipitation (in)']} in\n")
                else:
                    print("Warning: Missing date in historical weather entry.")
        else:
            print("Error: Required columns are missing in the historical weather data.")
    except Exception as e:
        print(f"Error loading historical weather data: {e}")

def load_forecast_weather():
    """Load and display forecast weather data from a CSV file."""
    try:
        forecast_weather = pd.read_csv('forecast_weather.csv')
        if all(col in forecast_weather.columns for col in ['Date', 'Predicted Temperature (°F)', 'Predicted Dew Point (°F)', 'Predicted Humidity (%)', 'Predicted Wind Speed (mph)', 'Predicted Pressure (in)', 'Predicted Precipitation (in)']):
            print("Forecast Weather Data:")
            for index, row in forecast_weather.iterrows():
                if pd.notna(row['Date']):
                    print(f"Date: {row['Date']}")
                    
                    # Ensure the predicted temperature data is in the expected format
                    pred_temp_values = row['Predicted Temperature (°F)'].split()
                    if len(pred_temp_values) >= 3:  # Check if we have at least 3 values
                        print(f"Predicted Max Temperature: {pred_temp_values[0]}°F")
                        print(f"Predicted Avg Temperature: {pred_temp_values[1]}°F")
                        print(f"Predicted Min Temperature: {pred_temp_values[2]}°F")
                    else:
                        print("Warning: Predicted temperature data is not in the expected format.")
                    
                    print(f"Predicted Dew Point: {row['Predicted Dew Point (°F)']}°F")
                    print(f"Predicted Humidity: {row['Predicted Humidity (%)']}%")
                    print(f"Predicted Wind Speed: {row['Predicted Wind Speed (mph)']} mph")
                    print(f"Predicted Pressure: {row['Predicted Pressure (in)']} in")
                    print(f"Predicted Precipitation: {row['Predicted Precipitation (in)']} in\n")
                else:
                    print("Warning: Missing date in forecast weather entry.")
        else:
            print("Error: Required columns are missing in the forecast weather data.")
    except Exception as e:
        print(f"Error loading forecast weather data: {e}")

def main_menu():
    """Display the main menu and handle user input."""
    while True:
        print("\nWeather Data Management System")
        print("1. Load Current Weather")
        print("2. Load Monthly Weather")
        print("3. Load Historical Weather")
        print("4. Load Forecast Weather")
        print("5. Exit")
        
        choice = input("Please select an option (1-5): ")
        
        if choice == '1':
            load_current_weather()
        elif choice == '2':
            load_monthly_weather()
        elif choice == '3':
            load_historical_weather()
        elif choice == '4':
            load_forecast_weather()
        elif choice == '5':
            print("Exiting the program. Goodbye!")
            break
        else:
            print("Invalid choice. Please select a valid option.")

if __name__ == "__main__":
    main_menu()


Weather Data Management System
1. Load Current Weather
2. Load Monthly Weather
3. Load Historical Weather
4. Load Forecast Weather
5. Exit


Please select an option (1-5):  1


Current Weather Data:
Date: 2024-11-01
Dew Point: 29°F
Humidity: 42%
Wind Speed: 7 mph
Pressure: 27.7 in
Precipitation: 0 in

Date: 2024-11-02
Dew Point: 34°F
Humidity: 42%
Wind Speed: 10 mph
Pressure: 27.5 in
Precipitation: 0 in

Date: 2024-11-03
Dew Point: 36°F
Humidity: 45%
Wind Speed: 22 mph
Pressure: 27.7 in
Precipitation: 0 in

Date: 2024-11-04
Dew Point: 27°F
Humidity: 37%
Wind Speed: 28 mph
Pressure: 27.8 in
Precipitation: 0 in

Date: 2024-11-05
Dew Point: 27°F
Humidity: 37%
Wind Speed: 23 mph
Pressure: 27.8 in
Precipitation: 0 in

Date: 2024-11-06
Dew Point: 21°F
Humidity: 27%
Wind Speed: 21 mph
Pressure: 27.9 in
Precipitation: 0 in

Date: 2024-11-07
Dew Point: 24°F
Humidity: 24%
Wind Speed: 20 mph
Pressure: 27.9 in
Precipitation: 0 in

Date: 2024-11-08
Dew Point: 25°F
Humidity: 30%
Wind Speed: 9 mph
Pressure: 27.9 in
Precipitation: 0 in

Date: 2024-11-09
Dew Point: 29°F
Humidity: 41%
Wind Speed: 7 mph
Pressure: 27.8 in
Precipitation: 0 in

Date: 2024-11-10
Dew Point: 31°F
Hum

Please select an option (1-5):  2


Error: Required columns are missing in the monthly weather data.

Weather Data Management System
1. Load Current Weather
2. Load Monthly Weather
3. Load Historical Weather
4. Load Forecast Weather
5. Exit


KeyboardInterrupt: Interrupted by user

In [25]:
import pandas as pd

def extract_max_value(dataframe, column_name):
    """Extract and print the maximum value for a given column."""
    try:
        if column_name in dataframe.columns:
            max_value = dataframe[column_name].max()
            print(f"Max {column_name}: {max_value}")
        else:
            print(f"Error: Column '{column_name}' not found in the data.")
    except Exception as e:
        print(f"An error occurred while extracting max value for {column_name}: {e}")

def load_current_weather():
    """Load and display max values from current weather data."""
    try:
        current_weather = pd.read_csv('CURRENT WEATHER REPORT.csv')
        print("\nCurrent Weather Data:")
        extract_max_value(current_weather, 'Temperature (°F)')
        extract_max_value(current_weather, 'Dew Point (°F)')
        extract_max_value(current_weather, 'Humidity (%)')
        extract_max_value(current_weather, 'Wind Speed (mph)')
        extract_max_value(current_weather, 'Pressure (in)')
        extract_max_value(current_weather, 'Precipitation (in)')
    except Exception as e:
        print(f"Error loading current weather data: {e}")

def load_monthly_weather():
    """Load and display max values from monthly weather data."""
    try:
        monthly_weather = pd.read_csv('Monthly weather report (2019 - 2024).csv')
        print("\nMonthly Weather Data:")
        extract_max_value(monthly_weather, 'Avg Temperature')
        extract_max_value(monthly_weather, 'Avg Humidity')
        extract_max_value(monthly_weather, 'Total Rainfall')
    except Exception as e:
        print(f"Error loading monthly weather data: {e}")

def load_historical_weather():
    """Load and display max values from historical weather data."""
    try:
        historical_weather = pd.read_csv('Historical weather report (2019 - 2024).csv')
        print("\nHistorical Weather Data:")
        extract_max_value(historical_weather, 'Temperature (°F)')
        extract_max_value(historical_weather, 'Dew Point (°F)')
        extract_max_value(historical_weather, 'Humidity (%)')
        extract_max_value(historical_weather, 'Wind Speed (mph)')
        extract_max_value(historical_weather, 'Pressure (in)')
        extract_max_value(historical_weather, 'Precipitation (in)')
    except Exception as e:
        print(f"Error loading historical weather data: {e}")

def load_forecast_weather():
    """Load and display max values from forecast weather data."""
    try:
        forecast_weather = pd.read_csv('forecast_weather.csv')
        print("\nForecast Weather Data:")
        extract_max_value(forecast_weather, 'Predicted Temperature (°F)')
        extract_max_value(forecast_weather, 'Predicted Dew Point (°F)')
        extract_max_value(forecast_weather, 'Predicted Humidity (%)')
        extract_max_value(forecast_weather, 'Predicted Wind Speed (mph)')
        extract_max_value(forecast_weather, 'Predicted Pressure (in)')
        extract_max_value(forecast_weather, 'Predicted Precipitation (in)')
    except Exception as e:
        print(f"Error loading forecast weather data: {e}")

def main_menu():
    """Display the main menu and handle user input."""
    while True:
        print("\nWeather Data Management System")
        print("1. Load Current Weather")
        print("2. Load Monthly Weather")
        print("3. Load Historical Weather")
        print("4. Load Forecast Weather")
        print("5. Exit")
        
        choice = input("Please select an option (1-5): ")
        
        if choice == '1':
            load_current_weather()
        elif choice == '2':
            load_monthly_weather()
        elif choice == '3':
            load_historical_weather()
        elif choice == '4':
            load_forecast_weather()
        elif choice == '5':
            print("Exiting the program. Goodbye!")
            break
        else:
            print("Invalid choice. Please select a valid option.")

if __name__ == "__main__":
    main_menu()


Weather Data Management System
1. Load Current Weather
2. Load Monthly Weather
3. Load Historical Weather
4. Load Forecast Weather
5. Exit


Please select an option (1-5):  1



Current Weather Data:
Max Temperature (°F): Max
Max Dew Point (°F): Max
Max Humidity (%): Max
Max Wind Speed (mph): Max
Max Pressure (in): Max
Max Precipitation (in): Total

Weather Data Management System
1. Load Current Weather
2. Load Monthly Weather
3. Load Historical Weather
4. Load Forecast Weather
5. Exit


KeyboardInterrupt: Interrupted by user